In [1]:
import pysmile
import pysmile_license
from pathlib import Path
from pysmile.learning import DataSet, EM
import itertools
from expdef import Experiment, Analytic, Result
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
fileName = "DBNfromAG_learned.xdsl"
dataFileName = "dbnLogs100.csv"
targetNodes = ["tomcatWebServer_bruteForce", "DMZ_scanIP", "historian_scanVuln", "IED1_DERfailure"]
fixedNodes = ["historianServer_remoteShellAND", "MMSclient1_AND52", "MMSserver1_NodeAND23", "historianServer_NodeOR1"]
numSlices = 100

algoTypeExact = False

In [3]:
net = pysmile.Network()
net.read_file(fileName)
net.set_slice_count(numSlices)
if not algoTypeExact:
    # Set default inference algorithm and parameters
    net.set_bayesian_algorithm(pysmile.BayesianAlgorithmType.EPIS_SAMPLING)
    #pysmile.EPISParams.num_state_big
    # Print default EPIS algorithm parameters
    episParams = net.get_epis_params()
    episParams.propagation_length = 20  # Example modification
    net.set_epis_params(episParams)
    print("Propagation length: ", episParams.propagation_length)
    print("Num state small:", episParams.num_state_small)
    print("Num state medium:", episParams.num_state_medium)
    print("Num state big:", episParams.num_state_big)
else:
    net.set_bayesian_algorithm(pysmile.BayesianAlgorithmType.LAURITZEN)

Propagation length:  20
Num state small: 5
Num state medium: 8
Num state big: 20


In [4]:
def plotDefinitions(net: pysmile.Network):
    nodeHandles = net.get_all_nodes()
    nodeIds = net.get_all_node_ids()
    for nodeHandle, nodeId in zip(nodeHandles, nodeIds):
        nodeDef = net.get_node_definition(nodeHandle)
        nodeOutcomes = net.get_outcome_ids(nodeHandle)
        print(f"Node ID: {nodeId}, Definition: {nodeDef}, Outcomes: {nodeOutcomes}")

In [5]:
from validator import Validator

validator: Validator = Validator(net, dataFileName, fixedNodes)
for targetNode in targetNodes:
    validator.addClassNode(targetNode)
validator.kFold(nFolds = 5)

Fold 1/5 -- Accuracy: 0.9820, Precision: 0.9919, Recall: 0.9830, F1-score: 0.9874, MCC: 0.9558
Fold 2/5 -- Accuracy: 0.9826, Precision: 0.9891, Recall: 0.9865, F1-score: 0.9878, MCC: 0.9578
Fold 3/5 -- Accuracy: 0.9806, Precision: 0.9865, Recall: 0.9860, F1-score: 0.9863, MCC: 0.9534
Fold 4/5 -- Accuracy: 0.9815, Precision: 0.9897, Recall: 0.9837, F1-score: 0.9867, MCC: 0.9562
Fold 5/5 -- Accuracy: 0.9824, Precision: 0.9865, Recall: 0.9881, F1-score: 0.9873, MCC: 0.9585
Average Accuracy: 0.9818 ± 0.0007
Average Precision: 0.9888 ± 0.0021
Average Recall: 0.9855 ± 0.0019
Average F1-score: 0.9871 ± 0.0005
Average MCC: 0.9564 ± 0.0018
